# メーカー商品調査 — GitHubからColabで実行

各セルの▶を上から順に押してください。初期設定はGEXの5件試運転です。ZIPのアップロードは不要です。

KeepaキーとDrive保存先はColabのシークレット（左の鍵マーク）で `KEEPA_API_KEY` と `DRIVE_PARENT_FOLDER_ID` を設定すると、次回から入力を省略できます。シークレットが未設定なら実行時の入力欄から受け取ります。

実行前に結果を保存するGoogleアカウントでColabを開いてください。


In [ ]:
# @title ① GitHubからプログラムを取得
import os, sys, subprocess, tempfile
from pathlib import Path
REPOSITORY_URL = "https://github.com/gyallen0130/gex-keepa-analysis.git"
GITHUB_BRANCH = "main"
WORKSPACE = Path(tempfile.mkdtemp(prefix="manufacturer-checker-"))
REPO_DIR = WORKSPACE / "repo"
subprocess.run(["git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH,
                REPOSITORY_URL, str(REPO_DIR)], check=True)
PROJECT_DIR = REPO_DIR / "manufacturer_checker"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_DIR / "requirements.txt")], check=True)
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
for module_name in list(sys.modules):
    if module_name.split('.')[0] in {'collectors','core','matchers','keepa','analysis','output','ui'}:
        del sys.modules[module_name]
commit = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
print("準備完了。実行コード:", commit[:12])


In [ ]:
# @title ② 調査の設定
SOURCE = "GEX 5件の試運転" # @param ["GEX 5件の試運転", "GEX公式 全商品", "Excel/CSV/JSONの商品一覧"]
MANUFACTURER = "GEX" # @param {type:"string"}
JAN_LIMIT = 5 # @param {type:"integer"}
BB_LIMIT = 3 # @param {type:"integer"}
MIN_OTHER_SHARE = 0.1 # @param {type:"number"}
SAVE_TO_DRIVE = True # @param {type:"boolean"}
SHEET_NAME = "" # @param {type:"string"}
HEADER_ROW = 1 # @param {type:"integer"}
# JAN_LIMIT=0 は全件。BB_LIMIT=0 はBB取得を省略。
if not MANUFACTURER.strip() or JAN_LIMIT < 0 or BB_LIMIT < 0 or HEADER_ROW < 1 or not 0 <= MIN_OTHER_SHARE <= 100:
    raise ValueError("メーカー名・件数・閾値を確認してください")
if SOURCE.startswith("GEX") and MANUFACTURER.strip() != "GEX":
    raise ValueError("GEX公式の調査はメーカー名をGEXにしてください")
if SOURCE == "GEX 5件の試運転" and (JAN_LIMIT == 0 or JAN_LIMIT > 5 or BB_LIMIT > 3):
    raise ValueError("試運転ではJANを1〜5件、BBを0〜3件にしてください")
print("設定:", SOURCE, "/ メーカー:", MANUFACTURER)
print("JAN:", "全件" if JAN_LIMIT == 0 else JAN_LIMIT, "/ BB上限:", BB_LIMIT)


In [ ]:
# @title ③ Google Driveの認証と保存先確認
from google.colab import userdata
from output.drive import build_colab_drive_service, validate_parent
from ui.colab_runner import parse_folder_id

def optional_secret(name):
    try:
        return str(userdata.get(name) or "").strip()
    except Exception:
        return ""

drive_service = None
if SAVE_TO_DRIVE:
    folder_value = optional_secret("DRIVE_PARENT_FOLDER_ID")
    if not folder_value:
        folder_value = input("保存先の親フォルダURLまたはID: ").strip()
    DRIVE_PARENT_FOLDER_ID = parse_folder_id(folder_value)
    drive_service = build_colab_drive_service()
    parent = validate_parent(drive_service, DRIVE_PARENT_FOLDER_ID)
    print("保存先:", parent.get("name"), "→", MANUFACTURER)
else:
    print("今回はColab内への保存のみです。")


In [ ]:
# @title ④ 商品データを準備
import json
from collectors.gex import GexCollector
from collectors.excel import ExcelCollector
from ui.smoke import collect_gex_sample
collector_errors, input_warnings = [], []
if SOURCE == "GEX 5件の試運転":
    sample = collect_gex_sample(limit=5)
    products, collector_errors = sample["products"], sample["errors"]
elif SOURCE == "GEX公式 全商品":
    print("GEX公式を巡回します。JAN件数の制限は次のKeepa照合に適用されます。")
    collector = GexCollector()
    products = collector.collect()
    collector_errors = collector.errors
else:
    from google.colab import files
    print("商品一覧のxlsx/csv/tsv/jsonファイルを1つ選んでください。")
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("商品一覧は1ファイルずつ指定してください")
    input_path = Path(next(iter(uploaded)))
    if input_path.suffix.lower() == ".json":
        products = json.loads(input_path.read_text(encoding="utf-8"))
        if not isinstance(products, list) or any(not isinstance(p,dict) for p in products):
            raise ValueError("JSONは商品辞書の配列にしてください")
    else:
        collector = ExcelCollector(input_path,MANUFACTURER,sheet_name=SHEET_NAME or None,header_row=HEADER_ROW)
        products = collector.collect()
        input_warnings = collector.warnings
if not products:
    raise RuntimeError("商品データが0件です。公式サイトまたは入力ファイルを確認してください")
print("準備できた商品行数:", len(products))
for p in products[:5]:
    print(p.get("product_name", ""), "JAN:", p.get("jan", ""))
print("商品取得エラー:", len(collector_errors), "/ 入力確認:", len(input_warnings))


In [ ]:
# @title ⑤ Keepa照合・Buy Box分析・Excel作成（APIトークンを消費します）
import getpass
from ui.colab_runner import run_research
_api_key = optional_secret("KEEPA_API_KEY") or getpass.getpass("Keepa APIキー（非表示）: ")
try:
    local_excel, diagnostic_path, report = run_research(
        products, _api_key, MANUFACTURER,
        jan_limit=None if JAN_LIMIT == 0 else JAN_LIMIT,
        bb_limit=BB_LIMIT, min_other_share=MIN_OTHER_SHARE,
        collector_errors=collector_errors, input_warnings=input_warnings)
finally:
    del _api_key
report["code_commit"] = commit
diagnostic_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
print("Excel作成:", local_excel.name)
print("照合商品:", report["matched_source_products"], "/", report["source_products"], "/ ASIN:", report["unique_asins"])
print("消費トークン:", report["tokens_consumed"])
print("JAN取得エラー:", report["jan_errors"], "/ BB取得エラー:", report["bb_errors"])
if not report["bb_received"]:
    print("今回はBBデータを取得していません。BB動作確認は未完了です。")


In [ ]:
# @title ⑥ メーカー別フォルダに保存（失敗したらこのセルだけ再実行）
from output.drive import upload_result
if SAVE_TO_DRIVE:
    if report.get("drive_status") == "VERIFIED":
        print("保存済み:", report.get("drive_link") or report.get("drive_file_id"))
    else:
        saved = upload_result(drive_service, local_excel, MANUFACTURER, DRIVE_PARENT_FOLDER_ID)
        report.update(drive_status="VERIFIED", drive_file_id=saved["id"], drive_link=saved.get("webViewLink"))
        diagnostic_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
        print("Drive保存確認済み:", saved.get("webViewLink") or saved["id"])
else:
    from google.colab import files
    files.download(str(local_excel))
if report["jan_errors"] or report["bb_errors"]:
    print("調査エラーが含まれます。ExcelのKeepa取得エラータブを確認してください。")


## 必要な場合：診断ファイルをダウンロード
このファイルにはKeepaキーを記録しません。エラーの確認を依頼するときに使えます。

In [ ]:
from google.colab import files
files.download(str(diagnostic_path))
